# EduSense — Ottawa Traffic Collisions
## 00 · Preprocessing
Nettoyage, feature engineering et export du dataset vers `collisions_clean.parquet`.

**Input :** `Traffic_Collision_Data.csv`  
**Output :** `collisions_clean.parquet`

In [12]:
import pandas as pd
import numpy as np
from pyproj import Transformer
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('Traffic_Collision_Data.csv')
raw_count = len(df)
print(f"Dimensions brutes : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
df.head(3)

Dimensions brutes : 94,406 lignes × 28 colonnes


,X,Y,X_Coordinate,Y_Coordinate,ID,Geo_ID,Accident_Year,Accident_Date,Location,Classification_Of_Accident,...,num_of_motorcycles,Max_injury,num_of_injuries,num_of_minimal,num_of_minor,num_of_major,num_of_fatal,Lat,Long,ObjectId
0,-8.449907e+06,5.674412e+06,351293.8071,5021841.110,2017--101,__3Z00RNB,2017,1/4/2017,MARCH RD btwn 280 S OF CARLING AVE/STATION RD ...,02 - Non-fatal injury,...,NaN,Minimal,1.0,1.0,NaN,NaN,NaN,45.334978,-75.906802,1
1,-8.432341e+06,5.664915e+06,363723.6849,5015276.436,2017--201,0004726,2017,1/5/2017,GREENBANK RD @ BERRIGAN DR/WESSEX RD (0004726),03 - P.D. only,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.274975,-75.749006,2
2,-8.427552e+06,5.687484e+06,366942.7019,5031143.575,2017--801,0006357,2017,1/23/2017,ALBERT ST @ BAY ST (0006357),02 - Non-fatal injury,...,NaN,Minimal,1.0,1.0,NaN,NaN,NaN,45.417468,-75.705989,3


### 1 · Nettoyage géographique

In [13]:
# Supprimer les lignes avec Lat=0 (coordonnée invalide — 72 lignes)
df = df[df['Lat'] != 0.0].copy()

# Filtre bbox Ottawa : Lat [44.9, 45.6], Long [-76.4, -75.2]
df = df[df['Lat'].between(44.9, 45.6) & df['Long'].between(-76.4, -75.2)].copy()
print(f"Lignes conservées après filtres géo : {len(df):,}")

# Projection UTM 18N (EPSG:32618) — distances en mètres, requis pour DBSCAN
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32618", always_xy=True)
df['utm_x'], df['utm_y'] = transformer.transform(df['Long'].values, df['Lat'].values)
print("Projection UTM 18N ajoutée (utm_x, utm_y)")

Lignes conservées après filtres géo : 94,334
Projection UTM 18N ajoutée (utm_x, utm_y)


### 2 · Nettoyage temporel

In [14]:
df['Accident_Date'] = pd.to_datetime(df['Accident_Date'], format='%m/%d/%Y', errors='coerce')
df = df[df['Accident_Date'].notna()].copy()

df['month']       = df['Accident_Date'].dt.month
df['day_of_week'] = df['Accident_Date'].dt.dayofweek   # 0=lundi
df['quarter']     = df['Accident_Date'].dt.quarter

df['saison'] = df['month'].map({
    12:'Hiver',1:'Hiver',2:'Hiver',
    3:'Printemps',4:'Printemps',5:'Printemps',
    6:'Été',7:'Été',8:'Été',
    9:'Automne',10:'Automne',11:'Automne'})
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

print("Distribution des saisons :")
print(df['saison'].value_counts())
print(f"\nAnnées présentes : {sorted(df['Accident_Year'].unique().tolist())}")
print("→ 2023 absent des données source")

Distribution des saisons :
saison
Hiver        28291
Automne      25720
Été          21085
Printemps    19238
Name: count, dtype: int64

Années présentes : [2017, 2018, 2019, 2020, 2021, 2022, 2024]
→ 2023 absent des données source


### 3 · Variables catégorielles

In [15]:
def parse_coded(series):
    """Extrait code numérique et libellé depuis 'XX - Libellé'."""
    code  = series.str.extract(r'^(\d+)')[0].astype(float)
    label = series.str.extract(r'^\d+\s*-\s*(.+)$')[0].str.strip()
    return code, label

coded_cols = {
    'Classification_Of_Accident':'classif',
    'Initial_Impact_Type':'impact',
    'Road_1_Surface_Condition':'surface',
    'Environment_Condition_1':'meteo',
    'Light':'lumiere',
    'Traffic_Control':'controle',
}
for orig, prefix in coded_cols.items():
    code, label = parse_coded(df[orig].fillna('00 - Unknown'))
    df[f'{prefix}_code']  = code
    df[f'{prefix}_label'] = label
    for suf in ['_label','_code']:
        col = f'{prefix}{suf}'
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].mode()[0])

print("Colonnes créées :")
for p in coded_cols.values():
    print(f"  {p}_code / {p}_label")

Colonnes créées :
  classif_code / classif_label
  impact_code / impact_label
  surface_code / surface_label
  meteo_code / meteo_label
  lumiere_code / lumiere_label
  controle_code / controle_label


### 4 · Feature gravité

In [16]:
df['severity_score'] = df['Classification_Of_Accident'].map({
    '03 - P.D. only':0,'04 - Non-reportable':0,
    '02 - Non-fatal injury':1,'01 - Fatal injury':2
}).fillna(0).astype(int)

df['injury_rank'] = df['Max_injury'].map({'Minimal':1,'Minor':2,'Major':3,'Fatal':4})

df['is_severe'] = (
    (df['severity_score']==2) | (df['injury_rank'].isin([3,4])).fillna(False)
).astype(int)

print("Distribution severity_score :")
for v, c in df['severity_score'].value_counts().sort_index().items():
    lbl = {0:'Matériel/NR',1:'Blessure',2:'Fatale'}[v]
    print(f"  {v} — {lbl:20s} : {c:,}")
print(f"\nCollisions graves (is_severe=1) : {df['is_severe'].sum():,} ({df['is_severe'].mean()*100:.1f}%)")

Distribution severity_score :
  0 — Matériel/NR          : 79,178
  1 — Blessure             : 14,986
  2 — Fatale               : 170

Collisions graves (is_severe=1) : 965 (1.0%)


### 5 · Usagers vulnérables

In [17]:
for col in ['num_of_pedestrians','num_of_bicycles','num_of_motorcycles']:
    df[col] = df[col].fillna(0).astype(int)
df['num_of_vehicles'] = df['num_of_vehicles'].fillna(df['num_of_vehicles'].median()).round().astype(int)

df['has_pedestrian'] = (df['num_of_pedestrians'] > 0).astype(int)
df['has_bicycle']    = (df['num_of_bicycles']    > 0).astype(int)
df['has_motorcycle'] = (df['num_of_motorcycles'] > 0).astype(int)
df['has_vulnerable'] = ((df['has_pedestrian'] | df['has_bicycle'] | df['has_motorcycle']) > 0).astype(int)

print(f"Piétons    : {df['has_pedestrian'].sum():,}")
print(f"Cyclistes  : {df['has_bicycle'].sum():,}")
print(f"Motos      : {df['has_motorcycle'].sum():,}")
print(f"Vulnérables total : {df['has_vulnerable'].sum():,} ({df['has_vulnerable'].mean()*100:.1f}%)")

Piétons    : 1,828
Cyclistes  : 1,662
Motos      : 791
Vulnérables total : 4,260 (4.5%)


### 6 · Features pour clustering

In [18]:
df['is_winter_surface']    = df['surface_label'].isin(['Loose snow','Packed snow','Ice','Slush']).astype(int)
df['is_winter_weather']    = df['meteo_label'].isin(['Snow','Freezing Rain','Drifting Snow']).astype(int)
df['is_winter_conditions'] = ((df['is_winter_surface'] | df['is_winter_weather']) > 0).astype(int)
df['is_dark']              = (df['lumiere_label'] == 'Dark').astype(int)
df['is_low_light']         = df['lumiere_label'].isin(['Dark','Dusk','Dawn']).astype(int)
df['has_signal']           = (df['controle_label'] == 'Traffic signal').astype(int)
df['no_control']           = (df['controle_label'] == 'No control').astype(int)
df['impact_simple']        = df['impact_label'].map({
    'Rear end':'arriere','Angle':'angle','Sideswipe':'lateral',
    'Turning movement':'virage','SMV other':'smv',
    'SMV unattended vehicle':'smv','Approaching':'face_a_face','Other':'autre'
}).fillna('autre')

print("Features clustering :")
for col in ['is_winter_conditions','is_low_light','has_signal','no_control']:
    print(f"  {col:25s} : {df[col].sum():,} ({df[col].mean()*100:.1f}%)")

Features clustering :
  is_winter_conditions      : 15,792 (16.7%)
  is_low_light              : 27,865 (29.5%)
  has_signal                : 38,245 (40.5%)
  no_control                : 43,635 (46.3%)


### 7 · Export Parquet

In [19]:
keep_cols = [
    'ID','Geo_ID','ObjectId',
    'Lat','Long','utm_x','utm_y',
    'Accident_Year','Accident_Date','month','day_of_week','quarter','saison','is_weekend',
    'Classification_Of_Accident','Max_injury',
    'classif_code','classif_label',
    'impact_code','impact_label','impact_simple',
    'surface_code','surface_label',
    'meteo_code','meteo_label',
    'lumiere_code','lumiere_label',
    'controle_code','controle_label',
    'severity_score','injury_rank','is_severe',
    'num_of_vehicles','num_of_pedestrians','num_of_bicycles','num_of_motorcycles',
    'num_of_injuries','num_of_minimal','num_of_minor','num_of_major','num_of_fatal',
    'has_pedestrian','has_bicycle','has_motorcycle','has_vulnerable',
    'is_winter_surface','is_winter_weather','is_winter_conditions',
    'is_dark','is_low_light','has_signal','no_control',
]
df_clean = df[keep_cols].copy()

print(f"Colonnes finales : {len(keep_cols)}")
print(f"Lignes finales   : {len(df_clean):,}")
print(f"Supprimées total : {raw_count - len(df_clean):,}")

nulls = df_clean[['Lat','Long','severity_score','lumiere_label','surface_label']].isnull().sum()
print("\nNulls colonnes clés :", "Aucun ✓" if nulls.sum()==0 else nulls[nulls>0])

df_clean.to_parquet('collisions_clean.parquet', index=False)
print("\n✓ Exporté → collisions_clean.parquet")

Colonnes finales : 52
Lignes finales   : 94,334
Supprimées total : 72

Nulls colonnes clés : Aucun ✓

✓ Exporté → collisions_clean.parquet


### 8 · Aperçu

In [20]:
df_clean = pd.read_parquet('collisions_clean.parquet')
pivot = df_clean.groupby('Accident_Year')['severity_score'].value_counts().unstack(fill_value=0)
pivot.columns = ['Matériel/NR','Blessure','Fatal']
pivot

,Matériel/NR,Blessure,Fatal
Accident_Year,,,
2017,11651,2745,28
2018,11854,2675,26
2019,13741,2699,25
2020,8297,1751,18
2021,7787,1897,23
2022,10446,1992,23
2024,15402,1227,27
